# CCTV VRT Restore (Google Colab GPU)

1. **Select Kernel → Colab → GPU** (extension `google.colab`)
2. Run cells in order
3. Upload a video when prompted, or set a Drive path

For fully automated terminal runs from Cursor, use:
`python tools/colab/run_vrt_colab.py --input Original/CUT/cut.mkv`

In [1]:
import torch, subprocess, sys
print('CUDA:', torch.cuda.is_available())
if torch.cuda.is_available():
    print('GPU:', torch.cuda.get_device_name(0))
    print('VRAM_GB:', round(torch.cuda.get_device_properties(0).total_memory / 1024**3, 1))

CUDA: True
GPU: NVIDIA GeForce RTX 3050 Laptop GPU
VRAM_GB: 4.0


In [2]:
# Install system + Python deps
!apt-get update -qq && apt-get install -y -qq ffmpeg > /dev/null
!pip install -q einops timm opencv-python-headless

'apt-get' is not recognized as an internal or external command,
operable program or batch file.


In [3]:
from pathlib import Path
import shutil

ROOT = Path('/content/CCTV')
VRT = ROOT / 'tools' / 'VRT'
ROOT.mkdir(parents=True, exist_ok=True)

if not (VRT / 'main_test_vrt.py').exists():
    !git clone --depth 1 https://github.com/JingyunLiang/VRT {VRT}
!pip install -q -r {VRT}/requirements.txt

# Pull restore script from this repo if present in Drive, else write expects upload via CLI
RESTORE = ROOT / '.cursor/skills/vrt-video-restoration/scripts/restore_video.py'
print('restore_video.py exists:', RESTORE.exists())

Cloning into '\content\CCTV\tools\VRT'...


restore_video.py exists: False


In [4]:
# Option A: upload from Cursor Colab CLI (preferred)
# Option B: interactive browser upload
from google.colab import files
from pathlib import Path

IN_DIR = Path('/content/input')
OUT_DIR = Path('/content/output')
IN_DIR.mkdir(exist_ok=True)
OUT_DIR.mkdir(exist_ok=True)

existing = list(IN_DIR.glob('*'))
if existing:
    INPUT = existing[0]
    print('Using existing:', INPUT)
else:
    print('Upload a CCTV video…')
    uploaded = files.upload()
    name = next(iter(uploaded))
    INPUT = IN_DIR / name
    Path(name).replace(INPUT) if Path(name).exists() else None
    # files.upload already writes to cwd; move if needed
    if Path(name).exists():
        Path(name).replace(INPUT)
    print('Saved:', INPUT)

ModuleNotFoundError: No module named 'google.colab'

In [ ]:
# If restore_video.py is missing, fetch from GitHub raw skill path is local-only.
# Paste/upload via Colab CLI, or mount Drive.
from pathlib import Path
import textwrap

RESTORE = Path('/content/CCTV/.cursor/skills/vrt-video-restoration/scripts/restore_video.py')
if not RESTORE.exists():
    raise SystemExit(
        'Missing restore_video.py. From Cursor terminal run:\n'
        '  python tools/colab/run_vrt_colab.py --input Original/CUT/cut.mkv --setup-only'
    )

OUTPUT = Path('/content/output') / f'{INPUT.stem}_colab.mkv'

!python {RESTORE} \
  --input {INPUT} \
  --output {OUTPUT} \
  --task 008_VRT_videodenoising_DAVIS \
  --sigma 20 \
  --tile 8 256 256 \
  --tile_overlap 2 20 20 \
  --chunk-frames 64 \
  --max-side 0 \
  --vrt-root {VRT} \
  --work-dir /content/work/vrt \
  --keep-work \
  --num-workers 2

print('Done:', OUTPUT, OUTPUT.stat().st_size if OUTPUT.exists() else 0)

In [ ]:
from google.colab import files
files.download(str(OUTPUT))